# Tree diagnostics: is the supermatrix any good?

One notebook for **all tree variants**, so they can be compared on the same page. It is executed once by the
`tree_diagnostics` rule after every alignment exists, and looks at the **concatenated alignments** (before any
tree is built). The tree side (fragmentation, support, RED) is in `tree_accuracy`; the choice of copy for Duplicated
BUSCO genes is in `copy_selection`.

Variants are labelled `derep/copies`:

| part | values | meaning |
|---|---|---|
| derep | `all` / `derep` | every genome, or only the dereplicated representatives (external genomes always kept) |
| copies | `single` / `multi` | Complete BUSCO genes only, or also one copy of each Duplicated gene |

A quick-check subset run (label printed below) uses far fewer genomes: do not compare its numbers with a full run.

**How the notebook is organised.** The *Key findings* just below are generated from this run's numbers (they are
computed at the end and moved up). Each section then shows a figure, a one-line **Takeaway** with the numbers, and
how to read the figure. What counts as OK / worth a look / a problem is set by the thresholds in section 0, so
change a number there to change a verdict. The closing section lists what to do about anything that was flagged.

1. **Occupancy** - does every genome have enough genes, or are some mostly gaps?
2. **Divergence and gene overlap** - are genomes of the same clade closer, and do they share more genes, than genomes of different clades?
3. **Per-gene separation** - does any gene fail to tell clades apart?
4. **Single vs multi copy** - do the extra Duplicated genes help?

## 0. Setup

Parameters are written by the `tree_diagnostics` rule into a JSON file (path in `NB_PARAMS`), so the notebook has
no hard-coded paths. The statistics live in `scripts/tree_diagnostics_lib.py` (unit-tested); this notebook loads the
data, calls them and draws the results.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
PARAMS = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, PARAMS["scripts_dir"])
import tree_diagnostics_lib as lib
import notebook_narrative as nn

OUT = Path(PARAMS["out_dir"])        # tables and PNG copies of the figures
OUT.mkdir(parents=True, exist_ok=True)
RANKS = ["phylum", "class", "order", "family", "genus"]
N_PAIRS = PARAMS["n_pairs"]          # genome pairs sampled per rank
MIN_PAIRS = 30                       # within-clade pairs with data needed to score a gene
VARIANTS = [v for v in PARAMS["variants"] if Path(v["alignment"]).exists()]
LABELS = [v["label"] for v in VARIANTS]
COLOURS = dict(zip(LABELS, plt.cm.tab10.colors))   # one colour per variant in every figure

print("Run     :", PARAMS["label"])
print("Variants:", ", ".join(LABELS))

### Thresholds

What counts as a problem. Each rule has a *look* value (worth a look) and a *bad* value (a problem). The numbers are
defaults chosen for eukaryotic BUSCO supermatrices of a few thousand genomes; change them here and every verdict below follows.

In [ ]:
SPARSE_GAP = 0.75                                    # a genome row with at least this share of gaps is "sparse"
SPARSE_SHARE = nn.Rule(look=0.02, bad=0.10)          # share of sparse genomes in a variant
MEDIAN_GENES = nn.Rule(look=0.75, bad=0.50, higher_is_worse=False)   # median genes per genome / genes in the alignment
SEPARATION = nn.Rule(look=0.90, bad=0.75, higher_is_worse=False)     # probability a within-clade pair is closer than a between-clade pair
JACCARD_GAP = nn.Rule(look=0.15, bad=0.30)           # within minus between gene-set similarity (order level)
GENE_LOW_SEPARATION = 0.80                           # a gene below this order-level separation is weak
WEAK_GENE_SHARE = nn.Rule(look=0.03, bad=0.10)       # share of genes that are weak
TIE = 0.005                                          # separation differences below this are noise
MULTI_WORSE = nn.Rule(look=0.15, bad=0.30)          # share of genes where multi separates clades clearly worse than single

FINDINGS = nn.Findings()

In [ ]:
def show_all(df, width=200):
    '''
    Prints an entire polars dataframe in the notebook output.

    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width (in characters) of the printed dataframe.
        Defaults to 200.
    '''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        cfg.set_fmt_str_lengths(width)
        print(df)


def finish(fig, name):
    '''
    Saves a figure as PNG next to the tables and shows it in the notebook.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        The figure to save.
    name : str
        File name without extension.
    '''
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

## 1. Load every alignment

For each variant: the genome x column matrix (`-` = gap), the gene boundaries, and the sampled genome pairs
used in sections 2-4. Pairs are sampled within and between clades at every rank, with the same seed for every
variant, and scored with `p-distance` (differing residues over columns where both genomes have a residue),
`jaccard` (shared genes / genes in either genome) and `shared_genes`.

In [ ]:
taxonomy = lib.read_taxonomy(PARAMS["taxonomy"])   # {genome: {rank: name}} database lineages

DATA = {}   # label -> everything computed for that variant
for variant in VARIANTS:
    ids, matrix = lib.load_alignment(variant["alignment"])
    partitions = lib.read_partitions(variant["partitions"])
    occupancy, present = lib.occupancy(ids, matrix, partitions)
    report = lib.rank_report(ids, matrix, present, partitions, taxonomy,
                             ranks=RANKS, n_pairs=N_PAIRS, seed=1)
    gene_rows = [{"variant": variant["label"], "rank": rank, "gene": gene,
                  "separation": separation, "within_pairs_with_data": n}
                 for rank in report for gene, separation, n in lib.gene_separation(report, partitions, rank)]
    DATA[variant["label"]] = {
        "ids": ids, "partitions": partitions,
        "occupancy": pl.DataFrame(occupancy).with_columns(pl.lit(variant["label"]).alias("variant")),
        "genomes_per_gene": present.sum(axis=0),
        "report": {rank: {kind: {k: report[rank][kind][k] for k in ("pdist", "jaccard", "shared_genes")}
                          for kind in ("within", "between")} | {"summary": report[rank]["summary"]}
                   for rank in report},
        "gene_separation": pl.DataFrame(gene_rows),
    }
    print(f"{variant['label']:14} {len(ids):6d} genomes x {matrix.shape[1]} columns, {len(partitions)} genes")
    del matrix

## 2. Occupancy: genes per genome and gaps

Each genome's row of the supermatrix: how many genes it has at all, and what fraction of the row is gap.

**How to read it.** Most genomes should have well over half of the genes. A long tail with few genes adds mostly
gaps to the supermatrix; such genomes are placed unstably and can look "misplaced". The workflow already drops
genomes below `alignment.min_genes_per_genome`; if many rows are still above ~90% gaps, raise that threshold or
look at those genomes' BUSCO completeness. `genomes per gene` should be high for every gene; a gene present in
few genomes contributes little.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for label in LABELS:
    occ = DATA[label]["occupancy"]
    kw = dict(histtype="step", linewidth=1.6, color=COLOURS[label], label=label)
    axes[0].hist(occ["genes"], bins=40, **kw)
    axes[1].hist(occ["gap_fraction"], bins=40, **kw)
    axes[2].hist(DATA[label]["genomes_per_gene"], bins=40, **kw)
axes[0].set_title("genes per genome")
axes[1].set_title("gap fraction of each genome's row")
axes[2].set_title("genomes per gene")
axes[0].legend(fontsize=8)
finish(fig, "occupancy")

occupancy_all = pl.concat([DATA[label]["occupancy"] for label in LABELS])
occupancy_all.write_csv(OUT / "occupancy.tsv", separator="\t")

total_genes = {label: len(DATA[label]["partitions"]) for label in LABELS}
sparse_share = {label: float((DATA[label]["occupancy"]["gap_fraction"] >= SPARSE_GAP).mean()) for label in LABELS}
median_share = {label: DATA[label]["occupancy"]["genes"].median() / total_genes[label] for label in LABELS}
s_sparse, t_sparse = nn.across(SPARSE_SHARE, sparse_share, lambda v: nn.pct(v, 1))
s_median, t_median = nn.across(MEDIAN_GENES, median_share, nn.pct)
FINDINGS.say(nn.worst([s_sparse, s_median]),
             f"The median genome has {t_median} of the alignment's genes; genomes whose row is at least {nn.pct(SPARSE_GAP)} gaps make up {t_sparse} of each variant.",
             evidence=f"sparse share {t_sparse}; median genes {t_median}",
             action="Raise alignment.min_genes_per_genome, or check BUSCO completeness of the sparsest genomes (listed below).")

The genomes with the fewest genes are the first candidates to inspect. The bar length is the number of genes;
the colour is the variant. (Full list: `occupancy.tsv`.) Sparse genomes add mostly gaps to the supermatrix, are placed
unstably and can look "misplaced" in `tree_accuracy`, so a genome listed here that is also a misplaced tip there is
most likely a data problem, not a real relationship.

In [ ]:
N_SHOWN = 20
fewest = occupancy_all.sort("genes").group_by("genome", maintain_order=True).first().head(N_SHOWN)
fig, axis = plt.subplots(figsize=(8, 0.3 * N_SHOWN + 1.5))
phylum = lambda genome: taxonomy.get(genome, {}).get("phylum", "unlabelled")
axis.barh(range(N_SHOWN), fewest["genes"], color=[COLOURS[v] for v in fewest["variant"]])
axis.set_yticks(range(N_SHOWN))
axis.set_yticklabels([f"{g}  ({phylum(g)})" for g in fewest["genome"]], fontsize=8)
axis.invert_yaxis()
axis.set_xlabel("genes present")
axis.set_title(f"{N_SHOWN} genomes with the fewest genes")
finish(fig, "fewest_genes")

lowest = fewest.head(3)
nn.show("**Takeaway.** Fewest genes: " + "; ".join(
    f"`{g}` ({phylum(g)}) has {n} of {max(total_genes.values())}" for g, n in zip(lowest["genome"], lowest["genes"])) + ".")

## 3. Do clades share genes, and are they closer in sequence?

For genome pairs at each rank, **within** = same clade name, **between** = different names. Lines are medians and
the bars the inter-quartile range.

* **p-distance** - within-clade distance should fall steadily from phylum to genus while between-clade distance
  stays high. Once between-clade distance stops growing with depth, p-distance is saturated and cannot order the
  deep divergences; the tree then leans on the substitution model.
* **separation** - probability that a within pair is closer than a between pair (1.0 = clades perfectly
  distinguishable, 0.5 = no information, below 0.5 = inverted). If it is high but the tree still fragments a clade,
  suspect the labels, a few misplaced genomes or the tree model rather than the data.
* **gene-set Jaccard / shared genes** - within above between means clade members have similar gene sets. That can be
  real, but it can also mean *missing* genes follow the clade, so genomes could group by what they lack. Read it
  with section 2.

In [ ]:
def rank_line(axis, key, title, ylabel):
    '''
    Draws the median (and inter-quartile range) of a pair statistic against rank for every variant.

    Parameters
    ----------
    axis : matplotlib.axes.Axes
        Axis to draw on.
    key : str
        Pair statistic: "pdist", "jaccard" or "shared_genes".
    title, ylabel : str
        Axis title and y label. Solid lines are within-clade pairs, dashed are between-clade pairs.
    '''
    x = np.arange(len(RANKS))
    for label in LABELS:
        for kind, style in (("within", "-"), ("between", "--")):
            q = np.array([np.nanquantile(np.asarray(DATA[label]["report"][r][kind][key], dtype=float), [0.25, 0.5, 0.75])
                          for r in RANKS if r in DATA[label]["report"]])
            xs = x[:len(q)]
            axis.errorbar(xs, q[:, 1], yerr=[q[:, 1] - q[:, 0], q[:, 2] - q[:, 1]], linestyle=style,
                          color=COLOURS[label], marker="o", markersize=3, capsize=2, linewidth=1.2,
                          label=f"{label} {kind}" if key == "pdist" else None)
    axis.set_xticks(x)
    axis.set_xticklabels(RANKS)
    axis.set_title(title)
    axis.set_ylabel(ylabel)


fig, axes = plt.subplots(2, 2, figsize=(13, 8))
rank_line(axes[0, 0], "pdist", "p-distance (shared sites)", "p-distance")
rank_line(axes[0, 1], "jaccard", "gene-set Jaccard", "shared / union of genes")
rank_line(axes[1, 0], "shared_genes", "genes shared by the pair", "genes")
for label in LABELS:
    sep = [DATA[label]["report"][r]["summary"]["separation_pdist"] for r in RANKS if r in DATA[label]["report"]]
    axes[1, 1].plot(range(len(sep)), sep, marker="o", color=COLOURS[label], label=label)
axes[1, 1].axhline(0.5, color="grey", linestyle=":")
axes[1, 1].set_xticks(range(len(RANKS)))
axes[1, 1].set_xticklabels(RANKS)
axes[1, 1].set_ylim(0.45, 1.01)
axes[1, 1].set_title("separation (within pair closer than between pair)")
axes[0, 0].legend(fontsize=7, ncol=2)
finish(fig, "rank_divergence")

rank_summary = pl.concat([pl.DataFrame([DATA[l]["report"][r]["summary"] for r in RANKS if r in DATA[l]["report"]])
                          .with_columns(pl.lit(l).alias("variant")) for l in LABELS])
rank_summary.write_csv(OUT / "rank_summary.tsv", separator="\t")

first = LABELS[0]
within_by_rank = rank_summary.filter(pl.col("variant") == first)
sep_order = {l: DATA[l]["report"]["order"]["summary"]["separation_pdist"] for l in LABELS if "order" in DATA[l]["report"]}
sep_genus = {l: DATA[l]["report"]["genus"]["summary"]["separation_pdist"] for l in LABELS if "genus" in DATA[l]["report"]}
s_o, t_o = nn.across(SEPARATION, sep_order)
s_g, t_g = nn.across(SEPARATION, sep_genus)
FINDINGS.say(nn.worst([s_o, s_g]),
             f"Genomes of the same clade are closer than genomes of different clades {t_o} of the time at order level and {t_g} at genus level (1.0 = always, 0.5 = no signal).",
             evidence=f"separation order {t_o}, genus {t_g}",
             action="Low separation with clean labels means the data cannot resolve the clades; check the labels, then the alignment.")
pd_in = within_by_rank["median_pdist_within"].to_list()
pd_out = within_by_rank["median_pdist_between"].to_list()
nn.show(f"**Takeaway.** In `{first}` the median within-clade p-distance falls from {pd_in[0]:.2f} at phylum to {pd_in[-1]:.2f} at genus, "
        f"while between-clade distance stays near {min(pd_out):.2f}-{max(pd_out):.2f}: it no longer grows with depth, so p-distance is saturated "
        f"between clades and the deep branches rest on the substitution model.")
jacc = {}
for l in LABELS:
    row = rank_summary.filter((pl.col("variant") == l) & (pl.col("rank") == "order"))
    if row.height:
        jacc[l] = float(row["median_jaccard_within"][0] - row["median_jaccard_between"][0])
s_j, t_j = nn.across(JACCARD_GAP, jacc)
FINDINGS.say(s_j, f"At order level, members of a clade share more of their gene set than non-members do (difference in Jaccard similarity {t_j}).",
             evidence=f"Jaccard gap {t_j}",
             action="A large gap means genomes could group by the genes they lack; compare with the occupancy section and consider a higher min_genes_per_genome.")

## 4. Per-gene separation

The same within/between comparison one gene at a time. A gene whose within-clade distances are not smaller than
its between-clade distances adds noise (saturated, badly aligned or paralogous) and is a candidate to drop. Genes
seen in fewer than `MIN_PAIRS` sampled within-clade pairs are left out because their value is noisy.

**How to read it.** A few genes slightly below the rest is normal. A clear low tail (several genes below the weak-gene
threshold, `GENE_LOW_SEPARATION`) suggests removing them and re-running. A uniformly low distribution points at the labels
or the alignment step. The heatmap shows the genes with the lowest mean separation at order level, for every variant.

In [ ]:
gene_separation = pl.concat([DATA[l]["gene_separation"] for l in LABELS])
gene_separation.write_csv(OUT / "gene_separation.tsv", separator="\t")
usable = gene_separation.filter((pl.col("within_pairs_with_data") >= MIN_PAIRS) & pl.col("separation").is_not_nan())

fig, axes = plt.subplots(1, len(RANKS), figsize=(3.2 * len(RANKS), 3.2), sharey=True)
for axis, rank in zip(axes, RANKS):
    for label in LABELS:
        values = usable.filter((pl.col("rank") == rank) & (pl.col("variant") == label))["separation"]
        axis.hist(values, bins=25, range=(0.5, 1.0), histtype="step", color=COLOURS[label], label=label)
    axis.set_title(rank)
    axis.set_xlabel("gene separation")
axes[0].set_ylabel("genes")
axes[0].legend(fontsize=7)
finish(fig, "gene_separation_distribution")

order_sep = usable.filter(pl.col("rank") == "order")
weak_share = {l: float((order_sep.filter(pl.col("variant") == l)["separation"] < GENE_LOW_SEPARATION).mean()) for l in LABELS
              if order_sep.filter(pl.col("variant") == l).height}
median_gene = {l: float(order_sep.filter(pl.col("variant") == l)["separation"].median()) for l in weak_share}
s_w, t_w = nn.across(WEAK_GENE_SHARE, weak_share, nn.pct)
_, t_m = nn.across(SEPARATION, median_gene)
FINDINGS.say(s_w, f"The median gene separates clades {t_m} of the time at order level; {t_w} of genes fall below {GENE_LOW_SEPARATION:.2f}.",
             evidence=f"weak genes {t_w}",
             action="Drop the weak genes (heatmap below, gene_separation.tsv) and re-run the alignment and tree.")

In [ ]:
N_GENES = 25
order = usable.filter(pl.col("rank") == "order")
worst_genes = (order.group_by("gene").agg(pl.col("separation").mean().alias("mean"))
               .sort("mean").head(N_GENES)["gene"].to_list())
grid = np.full((len(worst_genes), len(LABELS)), np.nan)
for j, label in enumerate(LABELS):
    lookup = dict(zip(*order.filter(pl.col("variant") == label).select("gene", "separation").to_dict(as_series=False).values()))
    grid[:, j] = [lookup.get(g, np.nan) for g in worst_genes]

fig, axis = plt.subplots(figsize=(1.2 * len(LABELS) + 3, 0.3 * N_GENES + 1.5))
image = axis.imshow(grid, aspect="auto", vmin=0.5, vmax=1.0, cmap="viridis")
axis.set_xticks(range(len(LABELS)))
axis.set_xticklabels(LABELS, rotation=45, ha="right")
axis.set_yticks(range(len(worst_genes)))
axis.set_yticklabels(worst_genes, fontsize=8)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        if not np.isnan(grid[i, j]):
            axis.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white")
plt.colorbar(image, label="separation")
axis.set_title(f"{N_GENES} lowest-separation genes (order level)")
finish(fig, "gene_separation_worst_genes")

nn.show("**Takeaway.** Lowest mean separation at order level: " + ", ".join(
    f"`{g}` ({m:.2f})" for g, m in order.group_by("gene").agg(pl.col("separation").mean().alias("m")).sort("m").head(3).iter_rows())
    + (f". All are above the weak-gene threshold ({GENE_LOW_SEPARATION:.2f}), so none is a candidate to drop."
       if float(order_sep.group_by("gene").agg(pl.col("separation").mean().alias("m"))["m"].min()) >= GENE_LOW_SEPARATION
       else f". Some are below the weak-gene threshold ({GENE_LOW_SEPARATION:.2f}): see gene_separation.tsv."))

## 5. Single vs multi copy

For each `derep` value the same genomes are used in the `single` and `multi` alignments, so each gene can be
compared with itself: every point is one gene, its separation in the `single` alignment on the x axis and in the
`multi` alignment on the y axis. Points above the diagonal are genes the extra Duplicated copies **improve**.

**How to read it.** A cloud on the diagonal means the extra copies change little. A cloud clearly above it says the
extra copies add real signal; clearly below says the copy chosen for Duplicated genes is adding paralogs. Differences in
the third decimal are noise. How the copy is chosen, and how often the choice matters, is in `copy_selection`.

In [ ]:
PAIRED = [d for d in dict.fromkeys(v["derep"] for v in VARIANTS)
          if f"{d}/single" in DATA and f"{d}/multi" in DATA]
COMPARED = ["order", "family"]

if not PAIRED:
    print("No derep value has both a single and a multi alignment.")
else:
    fig, axes = plt.subplots(len(PAIRED), len(COMPARED), figsize=(4.5 * len(COMPARED), 4.2 * len(PAIRED)), squeeze=False)
    for row, derep in enumerate(PAIRED):
        for col, rank in enumerate(COMPARED):
            pick = lambda c: (usable.filter((pl.col("variant") == f"{derep}/{c}") & (pl.col("rank") == rank))
                              .select("gene", pl.col("separation").alias(c)))
            both = pick("single").join(pick("multi"), on="gene")
            axis = axes[row, col]
            axis.scatter(both["single"], both["multi"], s=10, alpha=0.7)
            axis.plot([0.5, 1], [0.5, 1], color="grey", linestyle=":")
            better = float((both["multi"] > both["single"]).mean()) if both.height else float("nan")
            axis.set_title(f"{derep} / {rank}: multi better for {better:.0%} of {both.height} genes", fontsize=9)
            axis.set_xlabel("separation, single")
            axis.set_ylabel("separation, multi")
    finish(fig, "single_vs_multi")

better, worse, mean_change = {}, {}, {}
for derep in PAIRED:
    pick = lambda c: (usable.filter((pl.col("variant") == f"{derep}/{c}") & (pl.col("rank") == "order"))
                      .select("gene", pl.col("separation").alias(c)))
    both = pick("single").join(pick("multi"), on="gene")
    if both.height:
        diff = both["multi"] - both["single"]
        better[derep] = float((diff > TIE).mean())
        worse[derep] = float((diff < -TIE).mean())
        mean_change[derep] = float(diff.mean())
if worse:
    s_w2, t_w2 = nn.across(MULTI_WORSE, worse, nn.pct)
    _, t_b2 = nn.across(nn.Rule(1, 2), better, nn.pct)
    _, t_c = nn.across(nn.Rule(1, 2), mean_change, lambda v: f"{v:+.4f}")
    FINDINGS.say(s_w2, f"Compared with single-copy, adding one copy of each Duplicated gene separates clades clearly better (by more than {TIE}) for {t_b2} of genes and clearly worse for {t_w2}; "
                       f"the rest are within noise (mean change {t_c}).",
                 evidence=f"clearly better {t_b2}, clearly worse {t_w2}",
                 action="If multi is not clearly better, prefer the single-copy tree: it avoids any chance of paralogs. The choice of copy is analysed in `copy_selection`.")

## 6. Which copy of a Duplicated gene? (moved)

This notebook used to test the "top BUSCO score" rule here with a 3-mer similarity proxy. The aggregation now picks the copy by
alignment to the same gene in related genomes, and logs every decision, so the question is answered directly (and with a better
measure) in **`copy_selection`**: how often the choice differs from the top score, how much more peer-like the chosen copy is, and
which genes and genomes are affected.

## 7. Summary and what to do next

The table has one row per variant. Below it, every finding that was not fine, worst first, with the evidence and a suggested
action. Tables and PNG copies of the figures are written next to the executed notebook (`tables/tree_diagnostics/`).

In [ ]:
summary = pl.DataFrame([{
    "variant": label,
    "genomes": DATA[label]["occupancy"].height,
    "median_genes": DATA[label]["occupancy"]["genes"].median(),
    "rows_90pct_gaps": int((DATA[label]["occupancy"]["gap_fraction"] >= 0.9).sum()),
    "separation_order": round(DATA[label]["report"]["order"]["summary"]["separation_pdist"], 3),
    "separation_genus": round(DATA[label]["report"]["genus"]["summary"]["separation_pdist"], 3),
    "genes_below_threshold_order": int(usable.filter((pl.col("variant") == label) & (pl.col("rank") == "order")
                                                & (pl.col("separation") < GENE_LOW_SEPARATION)).height),
} for label in LABELS])
show_all(summary)

nn.show("### What to do next\n\n" + FINDINGS.actions())
pl.DataFrame(FINDINGS.table()).write_csv(OUT / "findings.tsv", separator="\t")

## Key findings

In [ ]:
nn.show(FINDINGS.verdict() + "\n\n" + FINDINGS.bullets() +
        "\n\n*Generated from this run's numbers; the thresholds are in section 0. Details and figures follow.*")